# 06 · Which factor matters, and how well risk can be ranked

**RQ4.** Given route, departure hour and month, how well can delay risk be ranked?

Three models, each with its own honest ceiling: sequential R² to rank factors, a
quantile regression for the tail, and a time-based holdout classifier for risk.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

In [ ]:
from team01 import models, viz

panel = clean.completed(io.load_flights())

ranking = models.incremental_r2(panel)
print(f"model R-squared {ranking.attrs['model_r2']} on {ranking.attrs['rows']:,} sampled flights")
ranking

## 6.1 The tail moves differently

A factor can be irrelevant on average and decisive at the ninetieth percentile, so
the same design is fitted at q = 0.9.

In [ ]:
quantile = models.quantile_regression(panel, quantile=0.9)
print(f"pseudo R-squared {quantile['pseudo_r2']}")
quantile["table"].head(8)

## 6.2 Risk classification on a time holdout

Logistic baseline against histogram gradient boosting. The split is chronological, so
no later month trains an earlier one. Both are reported: a reader needs to see how
much the tree model actually adds, and here it is little — five observable factors
rank risk weakly, and that is the finding.

In [ ]:
risk = models.risk_classifier(panel, threshold=60)
print(f"test set starts {risk['split']['actual_test_start']} "
      f"(requested {risk['split']['requested_start']}), "
      f"base rate {risk['base_rate']:.3%}")

pd.DataFrame({name: metrics for name, metrics in risk["models"].items()
              if name != "importance"}).T

In [ ]:
viz.fig_importance(pd.DataFrame(risk["models"]["importance"]))
viz.fig_hour_response(risk["hour_response"], threshold=60)